# CTM-Based Song Recommendation System

This notebook implements a music recommendation pipeline using Contextualized Topic Models (CTM), specifically the `CombinedTM`. It covers:

1. **Data Preprocessing**  
   Cleaning and tokenizing song lyrics for CTM input.  
2. **CTM Model Training**  
   Fitting the VAE to BoW counts + transformer embeddings.  
3. **Topic Labeling**  
   Generating interpretable topic names with LangChain + OpenAI.  
4. **Evaluation**  
   Computing topic coherence and diversity metrics.  
5. **Similarity Calculation**  
   Building song/artist vectors and measuring cosine similarity.  
6. **Diversity-Aware Recommendations**  
   Applying Maximal Marginal Relevance (MMR) for diverse results.



In [ ]:
# Core imports
from dotenv import load_dotenv
import openai
from langchain.prompts import PromptTemplate
from langchain.schema import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from scipy.spatial.distance import cosine
import re
from nltk.tokenize import RegexpTokenizer
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
import nltk
from sentence_transformers import SentenceTransformer
from contextualized_topic_models.utils.preprocessing import WhiteSpacePreprocessing
from contextualized_topic_models.utils.data_preparation import TopicModelDataPreparation
from contextualized_topic_models.models.ctm import CombinedTM
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import os
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# CTM and related libraries

# For preprocessing

# For similarity and recommendations

# For LLM topic labeling

# NLTK downloads
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)

print("📦 All libraries imported successfully!")

# Load OpenAI API key
load_dotenv(override=True)
openai_api_key = os.getenv("OPENAI_API_KEY")
if not openai_api_key:
    print("⚠️ OpenAI API key not found. LLM-based labeling will be disabled.")


In [ ]:
# Load and inspect the data
df = pd.read_csv(
    '../Lyrics_extraction/scraped_lyrics_no_metadata.csv', encoding='utf-8')

# Basic cleaning
df.dropna(subset=['lyrics'], inplace=True)
df = df[df['lyrics'].str.len() > 50].reset_index(drop=True)

print(f"📊 Dataset Overview:")
print(f"   Songs: {len(df):,}")
print(f"   Artists: {df['artist'].nunique():,}")
print(f"   Columns: {list(df.columns)}")
df.head()


In [ ]:
# ===== PREPROCESSING FOR CTM =====

def clean_lyrics_for_ctm(text):
    """Clean lyrics for CTM preprocessing."""
    if pd.isna(text) or not text:
        return ""
    text = str(text)
    text = re.sub(r'\[.*?\]', '', text)  # Remove structural markers
    text = re.sub(r'\n', ' ', text)      # Replace newlines with spaces
    text = re.sub(r'+', ' ', text)      # Normalize whitespace
    return text.strip().lower()


# Apply cleaning
print("🧹 Cleaning lyrics...")
df['lyrics_cleaned'] = df['lyrics'].apply(clean_lyrics_for_ctm)

# Prepare documents for CTM
documents = df['lyrics_cleaned'].tolist()
sp = WhiteSpacePreprocessing(documents, stopwords_language='english')
preprocessed_documents, unpreprocessed_corpus, vocab = sp.preprocess()

print(f"✅ Preprocessing complete:")
print(f"   Vocabulary size: {len(vocab):,}")

# Prepare contextual embeddings
print("🧠 Generating sentence embeddings...")
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
training_embeddings = embedding_model.encode(
    unpreprocessed_corpus, show_progress_bar=True)

# Create TopicModelDataPreparation object
tp = TopicModelDataPreparation()
training_dataset = tp.fit(text_for_contextual=unpreprocessed_corpus,
                          text_for_bow=preprocessed_documents, embeddings=training_embeddings)

print(f"✅ CTM data prepared!")


In [ ]:
# ===== CTM MODEL TRAINING =====

n_topics = 20  # Number of topics to discover

print(f"🚀 Training CombinedTM with {n_topics} topics...")

ctm = CombinedTM(bow_size=len(
    tp.vocab), contextual_size=training_embeddings.shape[1], n_components=n_topics, num_epochs=20)

print("⏳ This may take a few minutes...")
# Set verbose=True for more details
ctm.fit(training_dataset, n_samples=2000, verbose=False)

print("✅ CTM training completed!")


In [ ]:
# ===== TOPIC ANALYSIS AND EVALUATION =====

print("📊 Analyzing discovered topics...")

# Get topic-word distributions
topics = ctm.get_topic_lists(10)

print("🔝 Top 10 Topics and their keywords:")
for i, topic_words in enumerate(topics):
    print(f"   Topic {i:2d}: {', '.join(topic_words)}")

# Evaluate topic coherence and diversity
print("📈 Evaluating model performance...")

# Topic Coherence
topic_coherence = ctm.get_coherence(training_dataset, top_n_words=10)
avg_coherence = np.mean(list(topic_coherence.values()))
print(f"   Average Topic Coherence (UMass): {avg_coherence:.4f}")

# Topic Diversity
topic_diversity = ctm.get_topic_diversity()
print(f"   Topic Diversity: {topic_diversity:.4f} (higher is better)")

# Plotting coherence
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.barplot(x=list(topic_coherence.values()), y=list(
    topic_coherence.keys()), orient='h', palette='viridis')
plt.title('Topic Coherence (UMass)')
plt.xlabel('Coherence Score')

plt.tight_layout()
plt.show()


In [ ]:
# ===== LLM-ASSISTED TOPIC LABELING =====


def generate_topic_labels_with_llm(topics, max_topics=20):
    """Generate human-readable topic labels using LangChain + OpenAI GPT."""
    print("🤖 Generating human-readable topic labels with LLM...")

    if not openai_api_key:
        print("⚠️ OpenAI API key not found. Using fallback labels.")
        return {i: f"Topic {i}" for i in range(len(topics))}

    try:
        llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.3,
                         max_tokens=100, api_key=openai_api_key)
        print("   ✅ Connected to OpenAI GPT-3.5-turbo")
    except Exception as e:
        print(f"   ❌ Failed to connect to OpenAI: {e}")
        return {i: f"Topic {i}" for i in range(len(topics))}

    prompt_template = PromptTemplate.from_template(
        """You are a music expert. Based on these keywords for a song topic: '{topic_words}', 
        provide a concise, human-readable topic label (2-4 words). Topic Label:"""
    )

    topic_labels = {}
    for i, topic_words in enumerate(topics[:max_topics]):
        try:
            prompt = prompt_template.format(topic_words=", ".join(topic_words))
            response = llm([HumanMessage(content=prompt)])
            label = response.content.strip()
            topic_labels[i] = label
            print(f"   Topic {i:2d}: '{label}'")
        except Exception as e:
            print(f"   ❌ Failed to generate label for topic {i}: {e}")
            topic_labels[i] = f"Topic {i}"

    return topic_labels


# Generate labels
topic_labels = generate_topic_labels_with_llm(topics, max_topics=n_topics)
print("
      ✅ Topic labeling complete!")


In [ ]:
# ===== SONG AND ARTIST TOPIC DISTRIBUTIONS =====

print("📊 Calculating topic distributions for songs...")
song_topic_distributions = ctm.get_thetas(training_dataset, n_samples=2000)
df['topic'] = np.argmax(song_topic_distributions, axis=1)
df['topic_probability'] = np.max(song_topic_distributions, axis=1)
df['topic_distribution'] = list(song_topic_distributions)
print("✅ Song topic distributions calculated.")


def aggregate_artist_topic_distributions(df):
    """Aggregate song-level topic distributions to the artist level."""
    print("🎤 Aggregating distributions to artist level...")
    artist_distributions = {}_
    artist_info = {}_

    for artist, artist_df in df.groupby('artist'):
        artist_song_distributions = np.vstack(
            artist_df['topic_distribution'].values)
        weights = artist_df['topic_probability'].values
        artist_dist = np.average(
            artist_song_distributions, axis=0, weights=weights)
        artist_dist /= artist_dist.sum()  # Normalize

        artist_distributions[artist] = artist_dist
        artist_info[artist] = {
            'num_songs': len(artist_df),
            'top_topics': np.argsort(artist_dist)[-3:][::-1],
            'sample_songs': artist_df['song_title'].head(3).tolist()
        }
    print(f"✅ Aggregated for {len(artist_distributions)} artists.")
    return artist_distributions, artist_info


artist_topic_distributions, artist_info = aggregate_artist_topic_distributions(
    df)

# Show sample artist profile
sample_artist = list(artist_topic_distributions.keys())[0]
print(f"🎤 Sample Profile: {sample_artist}")
dist = artist_topic_distributions[sample_artist]
info = artist_info[sample_artist]
for topic_idx in info['top_topics']:
    label = topic_labels.get(topic_idx, f'Topic {topic_idx}')
    print(f"   {label}: {dist[topic_idx]:.3f}")


In [ ]:
# ===== RECOMMENDATION SYSTEM LOGIC =====


def maximal_marginal_relevance(query_dist, candidate_distributions, candidate_info, lambda_param=0.7, top_k=10):
    """MMR for diverse recommendations."""
    relevance_scores = {artist: 1 - cosine(query_dist, dist)
                        for artist, dist in candidate_distributions.items()}

    selected = []
    remaining = list(candidate_distributions.keys())

    if not remaining:
        return []

    first_artist = max(remaining, key=lambda x: relevance_scores[x])
    selected.append(first_artist)
    remaining.remove(first_artist)

    while len(selected) < top_k and remaining:
        mmr_scores = {}
        for candidate in remaining:
            relevance = relevance_scores[candidate]
            max_similarity = max(
                [1 - cosine(candidate_distributions[candidate], candidate_distributions[s]) for s in selected])
            mmr_score = lambda_param * relevance - \
                (1 - lambda_param) * max_similarity
            mmr_scores[candidate] = mmr_score

        next_artist = max(remaining, key=lambda x: mmr_scores[x])
        selected.append(next_artist)
        remaining.remove(next_artist)

    return [{'artist': artist, 'relevance': relevance_scores[artist]} for artist in selected]


class CTMRecommendationSystem:
    def __init__(self, artist_distributions, artist_info, topic_labels):
        self.artist_distributions = artist_distributions
        self.artist_info = artist_info
        self.topic_labels = topic_labels

    def recommend_artists(self, query_artist, strategy='balanced_mmr', top_k=8):
        if query_artist not in self.artist_distributions:
            return f"Artist '{query_artist}' not found."

        query_dist = self.artist_distributions[query_artist]
        candidates = {
            k: v for k, v in self.artist_distributions.items() if k != query_artist}
        candidate_info = {k: v for k,
                          v in self.artist_info.items() if k != query_artist}

        if strategy == 'similarity':
            recs = sorted(candidates.items(), key=lambda item: 1 -
                          cosine(query_dist, item[1]), reverse=True)
            recommendations = [{'artist': r[0], 'relevance': 1 -
                                cosine(query_dist, r[1])} for r in recs[:top_k]]
        else:  # MMR
            lambda_param = 0.4 if strategy == 'diverse' else 0.7
            recommendations = maximal_marginal_relevance(
                query_dist, candidates, candidate_info, lambda_param=lambda_param, top_k=top_k)

        return self.format_recommendations(query_artist, recommendations)

    def format_recommendations(self, query_artist, recommendations):
        output = [f"🎵 RECOMMENDATIONS FOR: {query_artist}"]
        output.append("=" * 60)
        for i, rec in enumerate(recommendations):
            artist = rec['artist']
            info = self.artist_info[artist]
            top_topic_idx = info['top_topics'][0]
            top_topic_label = self.topic_labels.get(
                top_topic_idx, f'Topic {top_topic_idx}')
            output.append(
                f"   {i+1}. {artist} (relevance: {rec['relevance']:.3f})")
            output.append(
                f"      {info['num_songs']} songs, main theme: {top_topic_label}")
        return "


".join(output)

# Create and test the system
rec_system = CTMRecommendationSystem(
    artist_topic_distributions, artist_info, topic_labels)
print("🚀 CTM Recommendation System ready!")

test_artist = 'Taylor Swift'  # Or any other artist in the dataset
if test_artist in rec_system.artist_distributions:
    print(rec_system.recommend_artists(
        test_artist, strategy='balanced_mmr', top_k=5))
else:
    print(f"Test artist '{test_artist}' not found. Please choose another.")


# # 🎯 Usage Guide
#
# ## How to Use
#
# ### 1. Get Recommendations
# ```python
# # Balanced recommendations (default)
# print(rec_system.recommend_artists("Drake", strategy='balanced_mmr', top_k=5))
#
# # Pure similarity recommendations
# print(rec_system.recommend_artists("Drake", strategy='similarity', top_k=5))
#
# # Diverse recommendations
# print(rec_system.recommend_artists("Drake", strategy='diverse', top_k=5))
# ```